# STT — 02 · Prétraitement

On suit **un seul clip** pas à pas : signal brut → signal à 16 kHz → log-Mel → texte encodé, en regardant à chaque étape la forme du tenseur.

## Étape 1 — Lire un clip et le convertir en tenseur

In [ ]:
import pandas as pd
import soundfile as sf
import torch

# On prend le premier clip d'entraînement de fr-FR
train_fr = pd.read_csv("data/fr-FR/train.tsv", sep="	")
clip = train_fr.iloc[0]
print("Fichier :", clip["path"])
print("Phrase :", clip["sentence"])

# Lecture : soundfile renvoie un tableau NumPy et la fréquence d'échantillonnage
audio, sr = sf.read(f"data/fr-FR/clips/{clip['path']}", dtype="float32")
print("Type :", type(audio), "| forme :", audio.shape, "| fréquence :", sr, "Hz")

# Conversion en tenseur PyTorch (partage la même mémoire, sans copie)
waveform = torch.from_numpy(audio)
print("Tenseur :", waveform.shape, waveform.dtype)
print("Durée :", len(waveform) / sr, "s")
print("Amplitude min / max :", waveform.min().item(), waveform.max().item())

## Étape 2 — Rééchantillonner à 16 kHz

`resample` applique d'abord un **filtre passe-bas** (anti-aliasing : supprime ce qui dépasse 8 kHz, la limite de Nyquist à 16 kHz), puis réduit le nombre d'échantillons.

In [2]:
import torchaudio.functional as AF

TARGET_SR = 16000  # décision de l'EDA, point 4

# Filtre passe-bas (anti-aliasing) puis réduction du nombre d'échantillons
waveform_16k = AF.resample(waveform, orig_freq=sr, new_freq=TARGET_SR)

print("Avant :", waveform.shape, "à", sr, "Hz")
print("Après :", waveform_16k.shape, "à", TARGET_SR, "Hz")
print("Rapport :", len(waveform) / len(waveform_16k))
print("Durée après :", len(waveform_16k) / TARGET_SR, "s")

Avant : torch.Size([223488]) à 48000 Hz
Après : torch.Size([74496]) à 16000 Hz
Rapport : 3.0
Durée après : 4.656 s


## Étape 3 — Spectrogramme log-Mel

**Décision** : fenêtre 25 ms (400 échantillons), pas 10 ms (160 échantillons), 80 bandes Mel — standard de la reconnaissance vocale, identique à Whisper (comparaison plus juste).

In [3]:
import torchaudio.transforms as T

N_FFT = 400        # taille de fenêtre : 400 échantillons = 25 ms à 16 kHz
HOP_LENGTH = 160   # pas : 160 échantillons = 10 ms -> 100 instants par seconde
N_MELS = 80        # nombre de bandes Mel

# Objet qui enchaîne STFT puis regroupement en bandes Mel
mel_transform = T.MelSpectrogram(
    sample_rate=TARGET_SR,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
)

mel = mel_transform(waveform_16k)
print("Spectrogramme Mel :", mel.shape)   # (bandes, instants)

# Logarithme ; le petit 1e-6 évite log(0) = -infini sur les silences parfaits
log_mel = torch.log(mel + 1e-6)

# On transpose pour avoir (instants, caractéristiques), comme dans un Transformer
features = log_mel.T
print("Entrée du modèle :", features.shape)
print("Min / max / moyenne / écart-type :",
      features.min().item(), features.max().item(),
      features.mean().item(), features.std().item())

Spectrogramme Mel : torch.Size([80, 466])
Entrée du modèle : torch.Size([466, 80])
Min / max / moyenne / écart-type : -13.815510749816895 6.605799198150635 -7.790489196777344 5.582396507263184


## Étape 4 — Normalisation par clip

**Décision** : normalisation **par clip** (moyenne 0, écart-type 1, bande par bande) — compense les différences de micro et de volume, utile pour le passage Common Voice → robot.

In [4]:
# Moyenne et écart-type de chaque bande Mel, calculés sur tous les instants de ce clip
mean = features.mean(dim=0, keepdim=True)   # forme (1, 80)
std = features.std(dim=0, keepdim=True)     # forme (1, 80)

# Centrer (moyenne 0) et réduire (écart-type 1), bande par bande.
# Le 1e-5 évite une division par 0 si une bande est constante dans tout le clip.
features_norm = (features - mean) / (std + 1e-5)

print("Forme de la moyenne :", mean.shape)
print("Forme après normalisation :", features_norm.shape)
print("Moyenne globale :", features_norm.mean().item())
print("Écart-type global :", features_norm.std().item())

Forme de la moyenne : torch.Size([1, 80])
Forme après normalisation : torch.Size([466, 80])
Moyenne globale : -2.7832555815621163e-08
Écart-type global : 0.9989376664161682


## Étape 5 — Encodage du texte

Chaque caractère reçoit un indice ; le **blank** prend l'indice **0** (valeur par défaut de `nn.CTCLoss`).

In [ ]:
import re
import unicodedata

# Mêmes règles que celles validées au point 2 de l'EDA
ALLOWED = set("abcdefghijklmnopqrstuvwxyz" "àâæçéèêëîïôœùûüÿ" " '-.,?!")

def normalize(text):
    text = unicodedata.normalize("NFC", text)
    text = text.lower()
    text = re.sub(r"[’‘ʼ´]", "'", text)
    text = re.sub(r"[;:…«»()\"—–]", " ", text)
    text = re.sub(r" +", " ", text).strip()
    return text

# Vocabulaire : le blank en position 0, puis les 49 caractères triés
BLANK = "_"
vocab = [BLANK] + sorted(ALLOWED)

# Deux dictionnaires : caractère -> indice (encoder) et indice -> caractère (décoder)
char_to_id = {char: i for i, char in enumerate(vocab)}
id_to_char = {i: char for i, char in enumerate(vocab)}

text = normalize(clip["sentence"])
target = torch.tensor([char_to_id[char] for char in text])

print("Taille du vocabulaire :", len(vocab))
print("Texte normalisé :", text)
print("Indices :", target)
print("Forme :", target.shape)

# Vérification : décoder les indices doit redonner exactement le texte
decoded = "".join(id_to_char[i] for i in target.tolist())
print("Décodage identique :", decoded == text)

## Vérification du module `stt/`

Le code validé est rangé dans `stt/text.py` et `stt/audio.py`. On vérifie qu'il donne exactement le même résultat que les cellules ci-dessus.

In [ ]:
from stt.audio import load_audio, log_mel
from stt.text import normalize, encode, decode

features_module = log_mel(load_audio(f"data/fr-FR/clips/{clip['path']}"))
target_module = encode(normalize(clip["sentence"]))

print("Forme :", features_module.shape)
# allclose : égalité à d'infimes erreurs d'arrondi près
print("Features identiques :", torch.allclose(features_module, features_norm, atol=1e-5))
print("Cible identique :", torch.equal(target_module, target))
print("Décodage :", decode(target_module.tolist()))

## Étape 6 — Le `Dataset` PyTorch

`stt/dataset.py` : `load_split` assemble les tableaux d'un découpage (fr-FR + fr-CA), `CommonVoiceDataset` renvoie le clip n° *i* prétraité à la demande.

In [7]:
from stt.dataset import load_split, CommonVoiceDataset

folders = ["data/fr-FR", "data/fr-CA"]
train_table = load_split(folders, "train")
dev_table = load_split(folders, "dev")

train_dataset = CommonVoiceDataset(train_table)
dev_dataset = CommonVoiceDataset(dev_table)
print("Clips train :", len(train_dataset))
print("Clips dev :", len(dev_dataset))

# On demande un clip au hasard, comme le fera l'entraînement
features, target = train_dataset[100]
print("Features :", features.shape, "| cible :", target.shape)

Clips train : 12982
Clips dev : 484
Features : torch.Size([495, 80]) | cible : torch.Size([64])


## Étape 7 — DataLoader et padding

`collate_batch` complète les clips d'un batch avec des 0 jusqu'à la longueur du plus long, et renvoie les vraies longueurs (pour le CTC et le masque d'attention).

In [8]:
from torch.utils.data import DataLoader
from stt.dataset import collate_batch

train_loader = DataLoader(
    train_dataset,
    batch_size=8,              # 8 clips par batch
    shuffle=True,              # ordre aléatoire à chaque époque
    collate_fn=collate_batch,  # notre fonction de padding
)

# On prend un seul batch pour regarder les formes
features, feature_lengths, targets, target_lengths = next(iter(train_loader))
print("Features :", features.shape)
print("Longueurs réelles :", feature_lengths)
print("Cibles :", targets.shape)
print("Longueurs des phrases :", target_lengths)

Features : torch.Size([8, 757, 80])
Longueurs réelles : tensor([596, 577, 627, 634, 400, 757, 695, 389])
Cibles : torch.Size([8, 87])
Longueurs des phrases : tensor([76, 68, 79, 79, 52, 86, 87, 49])
